In [0]:

from pyspark.sql import functions as F

RUTA_DATASET = "/Volumes/workspace/default/music_data/dataset.csv"
RUTA_TRACKS = "/Volumes/workspace/default/music_data/tracks_features.csv"

# BRONZE: cargar los dos datasets originales sin limpiarlos.
df_dataset_original = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("escape", '"')
    .csv(RUTA_DATASET)
)

df_tracks_original = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("escape", '"')
    .csv(RUTA_TRACKS)
)

# Conservar la procedencia de cada registro.
df_bronze_dataset = df_dataset_original.withColumn(
    "archivo_origen", F.lit("dataset.csv")
)

df_bronze_tracks = df_tracks_original.withColumn(
    "archivo_origen", F.lit("tracks_features.csv")
)

# Guardar las fuentes originales como tablas Delta.
(
    df_bronze_dataset.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_dataset_musica")
)

(
    df_bronze_tracks.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_tracks_features")
)

# Validar los resultados.
print("CAPA BRONZE TERMINADA")
print("Dataset original:", f"{df_bronze_dataset.count():,}", "filas")
print("Tracks features original:", f"{df_bronze_tracks.count():,}", "filas")
print("Tablas creadas:")
print("bronze_dataset_musica")
print("bronze_tracks_features")


CAPA BRONZE TERMINADA
Dataset original: 114,000 filas
Tracks features original: 1,204,025 filas
Tablas creadas:
bronze_dataset_musica
bronze_tracks_features


In [0]:

# BRONZE: integrar los dos datasets originales en una sola tabla.
# No se limpian ni se eliminan registros.

df_bronze_combinado = df_bronze_dataset.unionByName(
    df_bronze_tracks,
    allowMissingColumns=True
)

# Guardar el resultado integrado como una tabla Delta.
(
    df_bronze_combinado.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_dataset_combinado")
)

print("INTEGRACIÓN BRONZE TERMINADA")
print(f"Dataset 1: {df_bronze_dataset.count():,} filas")
print(f"Dataset 2: {df_bronze_tracks.count():,} filas")
print(f"Dataset integrado: {df_bronze_combinado.count():,} filas")
print(f"Columnas totales: {len(df_bronze_combinado.columns)}")

display(df_bronze_combinado.limit(10))


INTEGRACIÓN BRONZE TERMINADA
Dataset 1: 114,000 filas
Dataset 2: 1,204,025 filas
Dataset integrado: 1,318,025 filas
Columnas totales: 31


_c0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,archivo_origen,id,name,album,album_id,artist_ids,track_number,disc_number,year,release_date
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,1.01e-06,0.358,0.715,87.917,4,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.42,0.166,1,-17.235,1,0.0763,0.924,5.56e-06,0.101,0.267,77.489,4,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.21,0.0,0.117,0.12,76.332,4,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Soundtrack),Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.905,7.07e-05,0.132,0.143,181.74,3,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.443,2,-9.681,1,0.0526,0.469,0.0,0.0829,0.167,119.949,4,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
5,01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,Days I Will Remember,58,214240,False,0.688,0.481,6,-8.807,1,0.105,0.289,0.0,0.189,0.666,98.017,4,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
6,6Vc5wAMmXdKIAM7WUoEb7N,A Great Big World;Christina Aguilera,Is There Anybody Out There?,Say Something,74,229400,False,0.407,0.147,2,-8.822,1,0.0355,0.857,2.89e-06,0.0913,0.0765,141.284,3,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
7,1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,We Sing. We Dance. We Steal Things.,I'm Yours,80,242946,False,0.703,0.444,11,-9.331,1,0.0417,0.559,0.0,0.0973,0.712,150.96,4,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
8,0IktbUcnAGrvD03AWnz3Q8,Jason Mraz;Colbie Caillat,We Sing. We Dance. We Steal Things.,Lucky,74,189613,False,0.625,0.414,0,-8.7,1,0.0369,0.294,0.0,0.151,0.669,130.088,4,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null
9,7k9GuJYLp2AzqokyEdwEw2,Ross Copperman,Hunger,Hunger,56,205594,False,0.442,0.632,1,-6.77,1,0.0295,0.426,0.00419,0.0735,0.196,78.899,4,acoustic,dataset.csv,null,null,null,null,null,null,null,null,null


In [0]:

from pyspark.sql import functions as F

# SILVER: partir de las dos fuentes originales de Bronze.
df_dataset = spark.table("bronze_dataset_musica")
df_tracks = spark.table("bronze_tracks_features")

# Eliminar el índice que se generó al exportar el primer CSV.
if "_c0" in df_dataset.columns:
    df_dataset = df_dataset.drop("_c0")

# Homologar los nombres de las columnas identificadoras.
if "track_id" not in df_tracks.columns and "id" in df_tracks.columns:
    df_tracks = df_tracks.withColumnRenamed("id", "track_id")

if "track_name" not in df_tracks.columns and "name" in df_tracks.columns:
    df_tracks = df_tracks.withColumnRenamed("name", "track_name")

# Integrar las fuentes con nombres de columnas consistentes.
df_silver = df_dataset.unionByName(
    df_tracks,
    allowMissingColumns=True
)

filas_iniciales = df_silver.count()

# Eliminar duplicados exactos.
df_silver = df_silver.dropDuplicates()

# Conservar canciones con identificador y título válidos.
df_silver = df_silver.filter(
    F.col("track_id").isNotNull() &
    (F.trim(F.col("track_id")) != "") &
    F.col("track_name").isNotNull() &
    (F.trim(F.col("track_name")) != "")
)

# Validar duración y tempo sin modificar los datos de Bronze.
if "duration_ms" in df_silver.columns:
    df_silver = df_silver.filter(
        F.col("duration_ms").cast("double").isNotNull() &
        (F.col("duration_ms").cast("double") > 0)
    )

if "tempo" in df_silver.columns:
    df_silver = df_silver.filter(
        F.col("tempo").cast("double").isNotNull() &
        (F.col("tempo").cast("double") > 0)
    )

# Guardar la tabla Silver corregida.
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_dataset_musica")
)

filas_finales = df_silver.count()

print("SILVER CORREGIDO")
print(f"Filas iniciales: {filas_iniciales:,}")
print(f"Filas finales: {filas_finales:,}")
print(f"Filas eliminadas: {filas_iniciales - filas_finales:,}")
print(f"Columnas: {len(df_silver.columns)}")

print("\nRegistros conservados por fuente:")
display(
    df_silver.groupBy("archivo_origen")
    .count()
    .orderBy("archivo_origen")
)

display(df_silver.limit(10))


SILVER CORREGIDO
Filas iniciales: 1,318,025
Filas finales: 1,314,640
Filas eliminadas: 3,385
Columnas: 28

Registros conservados por fuente:


archivo_origen,count
dataset.csv,113392
tracks_features.csv,1201248


track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,archivo_origen,album,album_id,artist_ids,track_number,disc_number,year,release_date
3FRARAKLpHw3er9ZaTwaHO,JJ Heller,I Believe in You,I Believe in You,53,198786,False,0.534,0.359,0,-9.311,1,0.0296,0.449,1.2e-05,0.331,0.267,131.804,4,acoustic,dataset.csv,null,null,null,null,null,null,null
45jGOHwYKgsRYbAJ8DR61d,Motohiro Hata,青の光景,ひまわりの約束,61,308146,False,0.523,0.445,10,-6.359,1,0.0329,0.188,0.0,0.18,0.336,78.957,4,acoustic,dataset.csv,null,null,null,null,null,null,null
1wflx5YklLYToaIbXkrZSq,Kina Grannis,Somebody Loved,Somebody Loved,57,153105,False,0.612,0.107,0,-15.867,1,0.0302,0.936,0.0,0.109,0.637,78.393,4,acoustic,dataset.csv,null,null,null,null,null,null,null
0YhsOegsemgmT7ZHdWheUy,Bailey Jehl,Anyone,Anyone,53,172766,False,0.371,0.208,11,-13.167,1,0.0462,0.912,0.0,0.105,0.309,199.805,4,acoustic,dataset.csv,null,null,null,null,null,null,null
5YRDGgdOshUPA17m7yzY7p,Masayoshi Yamazaki,HOME,"One more time, One more chance",53,330200,False,0.55,0.408,11,-8.404,0,0.0318,0.719,0.0,0.11,0.331,129.152,4,acoustic,dataset.csv,null,null,null,null,null,null,null
0BhmtRzdjNV0bQ5veUOazN,Anna Hamilton,Self Help,Self Help,36,156923,False,0.778,0.445,7,-8.228,1,0.0468,0.45,1.28e-06,0.119,0.502,130.01,4,acoustic,dataset.csv,null,null,null,null,null,null,null
2xHKGx65VCsJgU83Bvh4j7,Bailey Jehl,Apologize,Apologize,51,181596,False,0.566,0.168,3,-12.483,1,0.0372,0.812,0.0,0.125,0.238,116.286,4,acoustic,dataset.csv,null,null,null,null,null,null,null
3h8uWt8HNyDNubPSEBgTPG,Zack Tabudlo,Yakap,Yakap,66,178947,False,0.738,0.39,8,-8.437,1,0.0334,0.409,0.0,0.107,0.547,113.979,4,acoustic,dataset.csv,null,null,null,null,null,null,null
7KqahQEtWTG2xPKE4Kriad,Masaharu Fukuyama,伝言,追憶の雨の中 (Original Version),27,238866,False,0.345,0.919,2,-8.154,1,0.0751,0.189,0.0,0.17,0.407,174.147,4,acoustic,dataset.csv,null,null,null,null,null,null,null
0Wqht20X3ZII115vByDReU,Ali Farka Touré,Savane,Savane,9,463920,False,0.764,0.282,4,-10.46,0,0.0652,0.77,0.0833,0.0592,0.504,135.736,4,afrobeat,dataset.csv,null,null,null,null,null,null,null


In [0]:

from pyspark.sql import functions as F

df_silver = spark.table("silver_dataset_musica")

print("REGISTROS POR FUENTE")
display(
    df_silver.groupBy("archivo_origen")
    .count()
)

print("VALIDACIÓN DE CAMPOS PRINCIPALES")
display(
    df_silver.select(
        F.count("*").alias("total_filas"),
        F.count("track_id").alias("ids_no_nulos"),
        F.count("track_name").alias("nombres_no_nulos"),
        F.count("duration_ms").alias("duraciones_no_nulas"),
        F.count("tempo").alias("tempos_no_nulos")
    )
)


REGISTROS POR FUENTE


archivo_origen,count
tracks_features.csv,1201248
dataset.csv,113392


VALIDACIÓN DE CAMPOS PRINCIPALES


total_filas,ids_no_nulos,nombres_no_nulos,duraciones_no_nulas,tempos_no_nulos
1314640,1314640,1314640,1314640,1314640


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# GOLD: preparar el catálogo final para el recomendador musical.
df_gold = spark.table("silver_dataset_musica")

# Columnas necesarias para el recomendador y el PCA.
CARACTERISTICAS = [
    "tempo", "energy", "danceability", "valence",
    "acousticness", "instrumentalness", "liveness",
    "speechiness", "loudness"
]

COLUMNAS_GOLD = [
    "track_id", "track_name", "artists", "duration_ms"
] + CARACTERISTICAS

# Comprobar que existan las columnas necesarias.
faltantes = [
    c for c in COLUMNAS_GOLD
    if c not in df_gold.columns
]

if faltantes:
    raise ValueError(f"Faltan columnas necesarias para Gold: {faltantes}")

# Conservar popularity si existe; si no, crearla con valores nulos.
if "popularity" not in df_gold.columns:
    df_gold = df_gold.withColumn(
        "popularity", F.lit(None).cast("double")
    )

# Convertir las variables numéricas a tipos adecuados.
COLUMNAS_NUMERICAS = [
    "duration_ms", "popularity"
] + CARACTERISTICAS

for columna in COLUMNAS_NUMERICAS:
    df_gold = df_gold.withColumn(
        columna,
        F.col(columna).cast("double")
    )

# Conservar únicamente las columnas necesarias.
df_gold = df_gold.select(
    "track_id", "track_name", "artists",
    "duration_ms", *CARACTERISTICAS, "popularity"
)

filas_antes_gold = df_gold.count()

# Eliminar registros incompletos en las variables que utiliza el PCA.
df_gold = df_gold.dropna(
    subset=[
        "track_id", "track_name", "artists",
        "duration_ms", *CARACTERISTICAS
    ]
)

# Validar duración, tempo e identificadores.
df_gold = df_gold.filter(
    (F.length(F.trim(F.col("track_id"))) >= 10) &
    (F.trim(F.col("track_name")) != "") &
    (F.trim(F.col("artists")) != "") &
    F.col("duration_ms").between(60000, 900000) &
    F.col("tempo").between(1, 250)
)

# Conservar una sola fila por track_id.
# Si hay varias versiones, priorizar la que tenga popularity disponible.
ventana = Window.partitionBy("track_id").orderBy(
    F.col("popularity").desc_nulls_last(),
    F.col("track_name").asc()
)

df_gold = (
    df_gold
    .withColumn("_orden", F.row_number().over(ventana))
    .filter(F.col("_orden") == 1)
    .drop("_orden")
)

# Guardar el catálogo final como tabla Delta.
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_catalogo_recomendador")
)

filas_finales_gold = df_gold.count()

print("CAPA GOLD TERMINADA")
print(f"Filas antes de preparar Gold: {filas_antes_gold:,}")
print(f"Canciones finales únicas: {filas_finales_gold:,}")
print(f"Registros descartados: {filas_antes_gold - filas_finales_gold:,}")
print(f"Columnas finales: {len(df_gold.columns)}")
print("Tabla creada: gold_catalogo_recomendador")

display(df_gold.limit(10))


CAPA GOLD TERMINADA
Filas antes de preparar Gold: 1,314,640
Canciones finales únicas: 1,250,009
Registros descartados: 64,631
Columnas finales: 14
Tabla creada: gold_catalogo_recomendador


track_id,track_name,artists,duration_ms,tempo,energy,danceability,valence,acousticness,instrumentalness,liveness,speechiness,loudness,popularity
000B6fUCVkSThxEbewDZ8r,Fight or Flight,['Haunt'],156331.0,155.97799999999995,0.927,0.419,0.257,5.85E-4,7.94E-5,0.182,0.131,-6.035,null
000GdygHXoIYjX1RCnjvOv,Precession of the Equinoxes,['Nocturnus AD'],275229.0,150.738,0.9840000000000001,0.179,0.131,1.1E-4,0.88,0.134,0.154,-7.025,null
000Iz0K615UepwSJ5z2RE5,Böxig Leise - Pig & Dan Remix,Paul Kalkbrenner;Pig&Dan,515360.0,119.997,0.56,0.686,0.108,0.00114,0.181,0.111,0.0462,-13.264,22.0
000KKVfYkBOjKfPIkXPCji,Stompin' at the Savoy,['Johnny Frigo'],241400.0,115.13,0.27899999999999997,0.6,0.66,0.895,0.74,0.141,0.0366,-18.096,null
000TVLMODj7klbrl92vucb,Nada,['Addys Mercedes'],205253.0,123.999,0.545,0.9740000000000001,0.897,0.635,4.53E-4,0.0877,0.0606,-9.896,null
000Ya33x13VsVxQR55T1VR,Latex Agglutination,['Naegleria Fowleri'],87600.0,99.94,0.846,0.5870000000000001,0.547,9.82E-5,0.8809999999999999,0.075,0.0509,-5.7570000000000014,null
000cYQJyEbqfUm6Ls0Xs4s,Rain,['Mochudi'],221854.0,143.958,0.508,0.76,0.491,0.159,0.004079999999999999,0.11,0.0597,-8.397,null
000lt40NM8MTvLGXaX6NIJ,You Don't Wanna Run Away,['Love Thieves'],213907.0,115.976,0.534,0.659,0.611,0.428,0.0575,0.184,0.0304,-8.306000000000001,null
000pL3uIUapdLjgfmTsu0J,"Keyboard Sonata in D Major, K.118/L.122/P.266","['Domenico Scarlatti', 'Duanduan Hao']",300280.0,119.801,0.0506,0.426,0.5429999999999999,0.9940000000000001,0.8859999999999999,0.0997,0.0407,-27.086,null
000pmTmXnTezoN9uzTcZgo,"Sotto falso nome (From ""Sotto falso nome"")","['Angèle Dubeau', 'La Pietà']",234960.0,124.34,0.0921,0.174,0.0325,0.951,0.946,0.10099999999999999,0.0457,-22.143,null


In [0]:

from pyspark.sql import functions as F

# Leer el catálogo Gold existente.
df_gold = spark.table("gold_catalogo_recomendador")

# Convertir listas de artistas a texto separado por punto y coma.
df_gold = df_gold.withColumn(
    "artists",
    F.regexp_replace(
        F.col("artists"),
        r"""['"]\s*,\s*['"]""",
        ";"
    )
)

# Eliminar corchetes y comillas sobrantes.
df_gold = df_gold.withColumn(
    "artists",
    F.regexp_replace(
        F.col("artists"),
        r"""[\[\]'"]""",
        ""
    )
)

# Normalizar espacios y valores vacíos.
df_gold = df_gold.withColumn(
    "artists",
    F.trim(
        F.regexp_replace(F.col("artists"), r"\s*;\s*", ";")
    )
)

# Actualizar Gold sin cambiar las demás columnas.
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_catalogo_recomendador")
)

print("GOLD ACTUALIZADO")
display(
    spark.table("gold_catalogo_recomendador")
    .select("track_name", "artists")
    .limit(10)
)


GOLD ACTUALIZADO


track_name,artists
BOP,Honey Oat
Revolution,Warsaw Poland Bros.
Vientos del Sur,Avalanch
Doing It All for You,Checkpoint Charlie
Blues Is A Feeling,Blues Company
Tief,Paul Kalkbrenner
Body Parts,CES Cru
The Scars That You Can't See - Live,Secret Sphere
"Concerto No. 51 for 2 equal bass viols, ""Rougeville"": (Ouverture) Rougeville",Monsieur de Sainte-Colombe;Les Voix Humaines
Ayla,Blue Condition
